# latinCy Lemmatization and POS- / Morphology-Tagging

The low accuracy of certain tags in the Ciceronian, Sallustian, and Caesarian corpora (as identified in `postagged-data-accuracy-dashboard.Rmd`) encourages me to explore alternatives to Stanza for some of the basic morphology. LAS (Labeled Attachment Score), a common metric for treebanks, was only 63.16% for the Perseus treebanks, a major source for Caesar and Cicero. This score means that only 63% of words had the same syntactic head and dependency relation in the test set compared to gold-standard data. While the accuracy for POS and morphology on individual words reaches a bit higher, from the 75-95% range, the syntactic information is crucial for this study.

As a result, we want to set Stanza up for success as much as possible. Stanza struggles with specific verb forms, mistagging first person perfect forms or first person passive forms as second person, and has inconsistencies with lemmatization. If latinCy has higher performance on lemmatization and POS/morphology, passing these processes on the latinCy first before tagging with dependency relations might show an improvement. [Stanza's depparse does take into account the previous processors.](https://stanfordnlp.github.io/stanza/depparse.html#start-with-pretagged-document). 

This notebook will create a function for passing a sentence to Stanza, trying both models, and creating a quick data frame as a report.

Let's create a function that accepts a sentence, and runs a number of tests. I don't know whether stanza and spaCy are deterministic, but I will run each one 100 times regardless to see whether there's any variation in the accuracy. Here's the structure I have in mind for each column in the resulting data, with each row being a new iteration of the parsing process:

> 1. % of POS tags the same in Stanza and latinCy parsing
> 2. % of lemmas the same in ""
> 3. % of feats the same in ""
> 4. % of words in Stanza doc matching lemmas in gold standard.
> 5. % of words in Stanza doc matching POS in gold standard
> 6. % of words in latinCy doc matching lemmas in gold standard
> 7. % of words in latinCy doc matching POS in gold standard

Feats will be a challenge because the `InflClass` feature is only in Stanza, not spaCy. We'll have to account for that. 

I know Stanza has functions for breaking down the UFeats here; let's adapt it. This function will wrap around it and remove `InflClass` for comparison. Returning a `dict` will also allow us to check for equivalency easily.

In [1]:
from stanza.models.common.vocab import CompositeVocab
from stanza.models.common.doc import Word
from spacy.tokens.token import Token

from typing import Union

def dict_without_infclass(word: Union[Word,]) -> dict:
      if type(word) is not Word and type(word) is not Token:
          raise ValueError(f"Expected stanza.models.common.doc.Word or spacy.tokens.token.Token, not {type(word)}")
      comp = CompositeVocab(sep="|", keyed=True)
      if type(word) is Word:
          feats = word.feats
          if feats is None:
              return ""
          return_value = comp.unit2parts(feats)
          try:
              return_value.pop("InflClass")
          except KeyError:
              pass
      if type(word) is Token:
          feats = word.morph.to_json()
          return_value = comp.unit2parts(feats)
      return return_value

Next, we need to create a gold-standard dictionary for our test sentence:

In [2]:
text = "Hac oratione apud suos habita atque omnium mentibus excitatis dat centurionibus negotium"
tokenized = text.split(" ")

gs = [
	     {
             "text":"hae",
             "lemma":"hic",
             "pos":"DET",
         },
         {
		     "text":"oratione",
		     "lemma":"oratio",
		     "pos":"NOUN",
         },
         {
		     "text":"apud",
		     "lemma":"apud",
		     "pos":"ADP",
         },
         {
		     "text":"suos",
		     "lemma":"suus",
		     "pos":"DET",
         },
         {
		     "text":"habita",
		     "lemma":"habeo",
		     "pos":"VERB",
         },
         {
		     "text":"atque",
		     "lemma":"atque",
		     "pos":"CCONJ",
         },
         {
		     "text":"omnium",
		     "lemma":"omnis",
		     "pos":"DET",
         },
         {
		     "text":"mentibus",
		     "lemma":"mens",
		     "pos":"NOUN",
         },
         {
		     "text":"excitatis",
		     "lemma":"excito",
		     "pos":"VERB",
         },
         {
		     "text":"dat",
		     "lemma":"do",
		     "pos":"VERB",
         },
         {
		     "text":"centurionibus",
		     "lemma":"centurio",
		     "pos":"NOUN",
         },
         {
		     "text":"negotium",
		     "lemma":"negotium",
		     "pos":"NOUN",
         },
]

Now, we need to run it many times and add to a `pandas` data frame:

In [7]:
import pandas as pd
import sys


def test_stanza(text=text):
    import stanza

    sys.path.append("/home/mdehass/PycharmProjects/corpus-caesarianum-authorship/process_perseus_texts")
    
    from latincy_processor_variants import LatincyPOS, LatincyLemmatizer
    # Download model
    stanza.download('la', processors='tokenize,mwt,lemma,pos,depparse')
    
    # Refactor this into a function!
    data = {
        "pos_equal":[],
        "lemmas_equal":[],
        "feats_equal":[],
        "st_lemma_gs":[],
        "st_pos_gs":[],
        "sp_lemma_gs":[],
        "sp_pos_gs":[],
    }
    
    import stanza
        
    sp_pipeline = stanza.Pipeline('la', processors={"lemma":"latincy", "pos":"latincy",}, download_method=None)
    st_pipeline = stanza.Pipeline('la', processors="tokenize,mwt,lemma,pos,depparse", download_method=None)
    
    sp_doc = [x for x in sp_pipeline(text).iter_words()]
    st_doc = [x for x in st_pipeline(text).iter_words()]
    
    equivalent_values = 0
    for index, word in enumerate(sp_doc):
        if word.upos == st_doc[index].upos:
            equivalent_values += 1
    
    data["pos_equal"].append(equivalent_values / len(sp_doc))
    
    equivalent_values = 0
    for index, word in enumerate(sp_doc):
        if word.lemma == st_doc[index].lemma:
            equivalent_values += 1
            
    data["lemmas_equal"].append(equivalent_values / len(sp_doc))
    
    equivalent_values = 0
    for index, word in enumerate(sp_doc):
        if dict_without_infclass(word) == dict_without_infclass(st_doc[index]):
            equivalent_values += 1
            
    data["feats_equal"].append(equivalent_values / len(sp_doc))
    
    equivalent_values = 0
    for index, word in enumerate(st_doc):
        if word.lemma == gs[index]["lemma"]:
            equivalent_values += 1
            
    data["st_lemma_gs"].append(equivalent_values / len(sp_doc))
    
    equivalent_values = 0
    for index, word in enumerate(st_doc):
        if word.upos == gs[index]["pos"]:
            equivalent_values += 1
            
    data["st_pos_gs"].append(equivalent_values / len(sp_doc))
    
    equivalent_values = 0
    for index, word in enumerate(sp_doc):
        if word.lemma == gs[index]["lemma"]:
            equivalent_values += 1
            
    data["sp_lemma_gs"].append(equivalent_values / len(sp_doc))
    
    equivalent_values = 0
    for index, word in enumerate(sp_doc):
        if word.upos == gs[index]["pos"]:
            equivalent_values += 1
            
    data["sp_pos_gs"].append(equivalent_values / len(sp_doc))
    
    sp_doc = sp_pipeline(text)
    st_doc = st_pipeline(text)
    
    from pprint import pprint
    pprint(sp_doc)
    pprint(st_doc)
        
    df = pd.DataFrame(data)
    
    print(df)

The results are mostly the same.

Stanza adds `PronType` as a feature, while spaCy does not.

Let's look at the features side-by-side.

In [4]:
#print(text)
#print(f"{"spaCy":<80} {"Stanza":<80}")
#for index, token in enumerate(sp_doc.sentences[0].words):
#    try:
#        print(f"{sp_doc.sentences[0].words[index].feats:<80} {st_doc.sentences[0].words[index].feats:<80}")
#    except TypeError:
#        print("\n")

In [5]:
# Let's try a new text

text = "quae res et cibi genere et cotidiana exercitatione et libertate vitae, quod a pueris nullo officio aut disciplina adsuefacti nihil omnino contra voluntatem faciunt, et vires alit et immani corporum magnitudine homines efficit"
tokenized = text.split(" ")

gs = [
    {
        "text":"quae",
        "lemma":"qui",
        "pos":"DET", # DET confirmed by searching for "qui\tDET" in the UD PROIEL conversion
    },
    {
        "text":"res",
        "lemma":"res",
        "pos":"NOUN", 
    },
    {
        "text":"et",
        "lemma":"et",
        "pos":"CCONJ", 
    },
    {
        "text":"cibi",
        "lemma":"cibus",
        "pos":"NOUN", 
    },
    {
        "text":"genere",
        "lemma":"genus",
        "pos":"NOUN",
    },
    {
        "text":"et",
        "lemma":"et",
        "pos":"CCONJ", 
    },
    {
        "text":"cotidiana",
        "lemma":"cotidianus",
        "pos":"ADJ", 
    },
    {
        "text":"exercitatione",
        "lemma":"exercitatio",
        "pos":"NOUN", 
    },
    {
        "text":"et",
        "lemma":"et",
        "pos":"CCONJ", 
    },
    {
        "text":"libertate",
        "lemma":"libertas",
        "pos":"NOUN", 
    },
    {
        "text":"vitae",
        "lemma":"vita",
        "pos":"NOUN", 
    },
    {
        "text":",",
        "lemma":",",
        "pos":"PUNCT", 
    },
    {
        "text":"quod",
        "lemma":"quod",
        "pos":"SCONJ", 
    },
    {
        "text":"a",
        "lemma":"ab",
        "pos":"ADP", 
    },
    {
        "text":"pueris",
        "lemma":"puer",
        "pos":"NOUN", 
    },
    {
        "text":"nullo",
        "lemma":"nullus",
        "pos":"DET", 
    },
    {
        "text":"officio",
        "lemma":"officium",
        "pos":"NOUN", 
    },
    {
        "text":"aut",
        "lemma":"aut",
        "pos":"CCONJ", 
    },
    {
        "text":"disciplina",
        "lemma":"disciplina",
        "pos":"NOUN", 
    },
    {
        "text":"adsuefacti",
        "lemma":"adsuefacio",
        "pos":"VERB", 
    },
    {
        "text":"nihil",
        "lemma":"nihil",
        "pos":"PRON", 
    },
    {
        "text":"omnino",
        "lemma":"omnino",
        "pos":"ADV", 
    },
    {
        "text":"contra",
        "lemma":"contra",
        "pos":"ADP", 
    },
    {
        "text":"voluntatem",
        "lemma":"voluntas",
        "pos":"NOUN", 
    },
    {
        "text":"faciunt",
        "lemma":"facio",
        "pos":"VERB", 
    },
    {
        "text":",",
        "lemma":",",
        "pos":"PUNCT", 
    },
    {
        "text":"et",
        "lemma":"et",
        "pos":"CCONJ", 
    },
    {
        "text":"vires",
        "lemma":"vis",
        "pos":"NOUN", 
    },
    {
        "text":"alit",
        "lemma":"alo",
        "pos":"VERB", 
    },
    {
        "text":"et",
        "lemma":"et",
        "pos":"CCONJ", 
    },
    {
        "text":"immani",
        "lemma":"immanis",
        "pos":"ADJ", 
    },
    {
        "text":"corporum",
        "lemma":"corpus",
        "pos":"NOUN", 
    },
    {
        "text":"magnitudine",
        "lemma":"magnitudo",
        "pos":"NOUN", 
    },
    {
        "text":"homines",
        "lemma":"homo",
        "pos":"NOUN", 
    },
    {
        "text":"efficit",
        "lemma":"efficio",
        "pos":"VERB", 
    },
    
]

In [6]:
test_stanza(text=text)

2026-07-24 10:37:50 INFO: Downloaded file to /home/mdehass/.cache/stanza/1.14.0/resources/resources.json
2026-07-24 10:37:50 INFO: Downloading these customized packages for language: la (Latin)...
| Processor | Package       |
-----------------------------
| tokenize  | ittb          |
| mwt       | ittb          |
| pos       | ittb_nocharlm |
| lemma     | ittb_nocharlm |
| depparse  | ittb_nocharlm |
| pretrain  | conll17       |

2026-07-24 10:37:50 INFO: File exists: /home/mdehass/.cache/stanza/1.14.0/resources/la/tokenize/ittb.pt
2026-07-24 10:37:50 INFO: File exists: /home/mdehass/.cache/stanza/1.14.0/resources/la/mwt/ittb.pt
2026-07-24 10:37:50 INFO: File exists: /home/mdehass/.cache/stanza/1.14.0/resources/la/pos/ittb_nocharlm.pt
2026-07-24 10:37:50 INFO: File exists: /home/mdehass/.cache/stanza/1.14.0/resources/la/lemma/ittb_nocharlm.pt
2026-07-24 10:37:50 INFO: File exists: /home/mdehass/.cache/stanza/1.14.0/resources/la/depparse/ittb_nocharlm.pt
2026-07-24 10:37:51 INFO: Fi

2026-07-24 10:38:08 INFO: Downloaded file to /home/mdehass/.cache/stanza/1.14.0/resources/resources.json
2026-07-24 10:38:09 INFO: Loading these models for language: la (Latin):
| Processor | Package       |
-----------------------------
| tokenize  | ittb          |
| mwt       | ittb          |
| pos       | latincy       |
| lemma     | latincy       |
| depparse  | ittb_nocharlm |

2026-07-24 10:38:09 INFO: Using device: cpu
2026-07-24 10:38:09 INFO: Loading: tokenize
2026-07-24 10:38:09 INFO: Loading: mwt
2026-07-24 10:38:09 INFO: Loading: pos
2026-07-24 10:38:10 INFO: Loading: lemma
2026-07-24 10:38:12 INFO: Loading: depparse
2026-07-24 10:38:15 INFO: Done loading processors!
2026-07-24 10:38:15 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-07-24 10:38:15 INFO: Downloaded file to /home/mdehass/.cache/stanza/1.14.0/resources/resources.json
2026-07-24 10:38:16 INFO: Loading these models for language: la (Latin):
| Processor | Package       |
-----------------------------
| tokenize  | ittb          |
| mwt       | ittb          |
| pos       | ittb_nocharlm |
| lemma     | ittb_nocharlm |
| depparse  | ittb_nocharlm |

2026-07-24 10:38:16 INFO: Using device: cpu
2026-07-24 10:38:16 INFO: Loading: tokenize
2026-07-24 10:38:16 INFO: Loading: mwt
2026-07-24 10:38:16 INFO: Loading: pos
2026-07-24 10:38:19 INFO: Loading: lemma
2026-07-24 10:38:19 INFO: Loading: depparse
2026-07-24 10:38:19 INFO: Done loading processors!


[
  [
    {
      "id": 1,
      "text": "quae",
      "lemma": "qui",
      "upos": "PRON",
      "feats": "Case=Nom|Gender=Fem|Number=Plur",
      "head": 2,
      "deprel": "det",
      "start_char": 0,
      "end_char": 4
    },
    {
      "id": 2,
      "text": "res",
      "lemma": "res",
      "upos": "NOUN",
      "feats": "Case=Nom|Gender=Fem|Number=Plur",
      "head": 11,
      "deprel": "obl",
      "start_char": 5,
      "end_char": 8
    },
    {
      "id": 3,
      "text": "et",
      "lemma": "et",
      "upos": "CCONJ",
      "head": 4,
      "deprel": "cc",
      "start_char": 9,
      "end_char": 11
    },
    {
      "id": 4,
      "text": "cibi",
      "lemma": "cibus",
      "upos": "NOUN",
      "feats": "Case=Gen|Gender=Masc|Number=Sing",
      "head": 5,
      "deprel": "nmod",
      "start_char": 12,
      "end_char": 16
    },
    {
      "id": 5,
      "text": "genere",
      "lemma": "genus",
      "upos": "NOUN",
      "feats": "Case=Abl|Gender=Neut|Numb

Spacy performed better here, as it didn't hallucinate a lemma for `adsuefacti`, `immanis`, and `exercitatione`, but drew correct ones from the dictionary.

The POS performance is roughly equivalent. Getting the lemma right is crucial, though, as it helps it match the lemmas in the original treebanks it's trained on. It makes sense the lemmatization is better with latinCy, as they use a two-step process. The first-pass tokenization uses a probabilistic model, but the second pass uses a ~1 million word dictionary to vet unambiguous forms. [See the section in their open access book on the subject.](https://latincy.github.io/latincy-book/lemmatization.html)